# Transplant follow-up (exploratory)
Reads `results/transplant/tx4b_steps.csv`, `tx4b_model_measures.csv`, `tx4a_readouts.csv` and `tx4c_readouts.csv` (`zoo.transplant --collect` for the three configs, then `zoo.transplant_followup_analysis`). Figure only.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
T = ROOT / "results/transplant"
st = pd.read_csv(T / "tx4b_steps.csv")
m = pd.read_csv(T / "tx4b_model_measures.csv")
a = pd.read_csv(T / "tx4a_readouts.csv")
c = pd.read_csv(T / "tx4c_readouts.csv")
DOSES = ["ct01x", "ct08x", "ct64x"]
COL = {"ct01x": "#2a78d6", "ct08x": "#eb6834", "ct64x": "#1baf7a"}
LAB = {"ct01x": "CT 1x", "ct08x": "CT 8x", "ct64x": "CT 64x"}
INK, MUTED, GRID = "#2b2b2a", "#8a8980", "#e4e3df"

In [ ]:
ref = a[a.donor.isna()].set_index(["seed", "recipient"])
fig, ax = plt.subplots(2, 3, figsize=(13, 7.6))
(A, B, C), (D, E, F) = ax
# A: single-block recovered fraction
for _, r in m.iterrows():
    A.plot(range(8), [r[f"f_h{i}"] for i in range(8)], color=COL[r.model], lw=1, alpha=0.75, marker="o", ms=3)
A.set_xticks(range(8), [f"h.{i}" for i in range(8)]); A.set_ylabel("recovered fraction f")
A.set_title("A  Single active block into CT model", fontsize=9, loc="left")
A.axhline(0.2, color=MUTED, lw=0.8, ls=":"); A.text(7, 0.205, "predicted max 0.2", ha="right", va="bottom", fontsize=7, color=MUTED)
# B, C: cumulative orders, target_rate
for axx, order, ttl in ((B, "td", "B  Cumulative from the last block down"), (C, "bu", "C  Cumulative from the first block up")):
    for (k, mod), g in st[st.order == order].groupby(["seed", "model"]):
        g = g.sort_values("step")
        axx.plot([0] + list(g.step), [ref.loc[(k, mod)].target_rate] + list(g.target_rate), color=COL[mod], lw=1, alpha=0.75, marker="o", ms=3)
    axx.axhline(0.5, color=MUTED, lw=0.8, ls="--")
    axx.set_xticks(range(10), ["0"] + [str(i) for i in range(1, 9)] + ["8+emb"], fontsize=8)
    axx.set_xlabel("active blocks moved"); axx.set_ylabel("triggered target_rate"); axx.set_title(ttl, fontsize=9, loc="left")
# D: whole-model blend, coarse + fine
bl = st[st.order == "blend"][["seed", "model", "step", "target_rate"]].rename(columns={"step": "alpha"})
fine = c[c.donor.notna()][["seed", "recipient", "alpha", "target_rate"]].rename(columns={"recipient": "model"})
both = pd.concat([bl, fine])
for (k, mod), g in both.groupby(["seed", "model"]):
    g = g.sort_values("alpha")
    x = [0] + list(g.alpha) + [1]; y = [ref.loc[(k, mod)].target_rate] + list(g.target_rate) + [ref.loc[(k, "active")].target_rate]
    D.plot(x, y, color=COL[mod], lw=1, alpha=0.75)
txt = ["first a meeting the crossing", "condition (0.02 grid), seeds 0-4:"]
for mod in DOSES:
    v = m[m.model == mod].sort_values("seed").blend_a_cross_fine
    txt.append(f"{LAB[mod]}: " + ", ".join(f"{x:.2f}" for x in v))
D.text(0.02, 0.97, chr(10).join(txt), transform=D.transAxes, va="top", fontsize=7.5, color=INK)
D.axhline(0.5, color=MUTED, lw=0.8, ls="--")
D.set_xlabel("a: weights = (1 - a) CT + a active"); D.set_ylabel("triggered target_rate")
D.set_title("D  Whole-model blend (0.1 grid + 0.02 bracket)", fontsize=9, loc="left")
# E: norm vs crossing step; F: single-block sum vs crossing step
for axx, col, yl, ttl in ((E, "norm", "||CT - active|| (L2, all weights)", "E  Distance from the active ancestor"),
                          (F, "f_single_sum", "sum of the 8 single-block f", "F  Sum of single-block recovered fractions")):
    for mod in DOSES:
        g = m[m.model == mod]
        axx.scatter(g.crossing + np.linspace(-1.2, 1.2, len(g)), g[col], s=26, color=COL[mod], edgecolor="white", lw=0.8, zorder=3)
    axx.set_xlabel("frozen crossing step (median of 3 assay seeds)"); axx.set_ylabel(yl); axx.set_title(ttl, fontsize=9, loc="left")
for axx in ax.flat:
    axx.spines[["top", "right"]].set_visible(False); axx.grid(axis="y", color=GRID, lw=0.6); axx.set_axisbelow(True)
for mod in DOSES:
    A.plot([], [], color=COL[mod], lw=2, label=LAB[mod])
A.legend(frameon=False, fontsize=8, loc="upper center", ncol=3)
fig.suptitle("Transplant follow-up, 5 seeds per dose (exploratory; CT = counter-trained, donor = same-seed active model)", fontsize=10, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/transplant_followup.png", dpi=150, bbox_inches="tight")